In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog
from scipy.spatial import ConvexHull
from scipy.spatial import HalfspaceIntersection
import sympy as sp
from fractions import Fraction
from IPython.display import Math

# README

Дана автоматизація реалізує метод декомпозиції Корнаї-Ліптака для обмеженого класу задач лінійного програмування виду

$$
\begin{cases}
F=(F_1 x_1+F_2 x_2+F_3 x_3+F_4 x_4)->min; \\
A_{b1}x_1+A_{b2}x_2+A_{b3}x_3+A_{b4}x_4 \leq b_{bound};\\
A_{11}x_1+A_{12}x_2<=b_1; \\
A_{21}x_1+A_{22}x_2<=b_2; \\
A_{31}x_3+A_{32}x_4<=b_3; \\
A_{41}x_3+A_{42}x_4<=b_4; \\
x_j\geq0,j=\overline{1,4}
\end{cases}
$$

In [ ]:
F=np.array([4,1,2,3]) # np.array([F_1,F_2,F_3,F_4])
A_bound=np.array([2,1,1,2]) # np.array([A_b1,A_b2,A_b3,A_b4])
b_bound=21 # b_bound
A1=np.array([
    [2,-1],# [A_11,A_12]
    [1,2],# [A_21,A_22]
    [-1,0],# Не змінювати, відповідає за обмеження x_j>0
    [0,-1]# Не змінювати, відповідає за обмеження x_j>0
])
b1=np.array(
    [4,22,0,0]# [b_1,b_2,0,0]. нулі відповідають за обмеження x_1>0, x_2>0
)
A2=np.array([
    [3,-1],# [A_31,A_32]
    [-1,2],# [A_41,A_42]
    [-1,0],# Не змінювати, відповідає за обмеження x_j>0
    [0,-1]# Не змінювати, відповідає за обмеження x_j>0
])
b2=np.array(
    [6,8,0,0]# [b_3,b_4,0,0]. нулі відповідають за обмеження x_1>0, x_2>0
)
def text_sign(a):
    if a>=0:
        return "+"
    else:
        return ""
def st(x):
    return Fraction(x).limit_denominator() 

In [98]:
temp_text=fr"""
    \begin{{cases}}
    max({F[0]}x_1{text_sign(F[1])}{F[1]}x_2)=f_1(y_1)\\
    {A_bound[0]}x_1{text_sign(A_bound[1])}{A_bound[1]}x_2 \leq y_1\\ 
    {A1[0][0]}x_1{text_sign(A1[0][1])}{A1[0][1]}x_2 \leq {b1[0]}\\ 
    {A1[1][0]}x_1{text_sign(A1[1][1])}{A1[1][1]}x_2 \leq {b1[1]}\\
    x_1 \geq 0,  x_2 \geq 0, y_1 \geq 0
    \end{{cases}}
"""
display(Math(r"\text{Запишемо 1-у блочну задачу}"))
display(Math(temp_text))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

# Block 1

In [99]:
F1=F[0:2]
Ab_1=A_bound[0:2]
coeffs_1=np.array([F1[i]/Ab_1[i] for i in range(len(F1))])
ind_1=np.argmax(coeffs_1)# яку з двох осей вибрали 0->OX, 1->OY
hsp1=np.hstack((A1,-1*b1.reshape(-1,1)))
interior_point=np.array([0.01,0.01])
hsp1_points=HalfspaceIntersection(halfspaces=hsp1, interior_point=interior_point)
# print(hsp1_points.intersections)
lim_OX1=0
lim_OY1=0
hp1=list(np.zeros_like(hsp1_points.intersections))
for p in range(len(hsp1_points.intersections)):
    hp1[p][0]=Fraction(hsp1_points.intersections[p][0]).limit_denominator()
    hp1[p][1]=Fraction(hsp1_points.intersections[p][1]).limit_denominator()
hp1=np.array(hp1)
for p in hp1:
    if p[0]==0 and p[1]!=0:
        lim_OY1=p
    if p[0]!=0 and p[1]==0:
        lim_OX1=p
H=hsp1
V=hp1
hull = ConvexHull(V)
eps=1e-9
ordered_vertices = V[hull.vertices]
edges = []
m = H.shape[0]

for i in range(len(ordered_vertices)):
    p1 = ordered_vertices[i]
    p2 = ordered_vertices[(i+1) % len(ordered_vertices)]

    for j in range(m):
        a, b, c = H[j]
        if abs(a*p1[0] + b*p1[1] + c) < eps and \
           abs(a*p2[0] + b*p2[1] + c) < eps:
            edges.append({
                "edge": (p1, p2),
                "constraint_index": j,
                "line": H[j]
            })
            break
#display(edges)
display(Math(r"\text{Розв'язуємо її графоаналітично}"))
if ind_1==0:
    display(Math(r"\text{Тобто масимум досягається спочатку на вісі абсцис } x_1"))
    display(Math(fr"x_{{2}} = 0,\ \ x^0_1={lim_OX1[0]},\ \ y_1={Ab_1[ind_1]}x^0_1={Ab_1@lim_OX1}"))
    c=Fraction(Ab_1@lim_OX1).limit_denominator()
    print(c)
else:
    display(Math(r"\text{Тобто масимум досягається спочатку на вісі ординат } x_2"))
    display(Math(fr"x_{{1}} = 0,\ \ x^0_2={lim_OY1[1]},\ \ y_1={Ab_1[ind_1]}x^0_2={Ab_1@lim_OY1}"))
    c=Fraction(Ab_1@lim_OY1).limit_denominator()

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

4


In [100]:
res_1=linprog(-Ab_1, A_ub=A1, b_ub=b1)
f1=[]
ders1=[]
ceilings1=[0]
x12=[]
# print(res_1.slack)
f1.append((fr"{Fraction(F1[ind_1]/Ab_1[ind_1]).limit_denominator()}*y_1;\ \ 0 \leq y_1 \leq {c}"))
if len(hsp1_points.intersections)>3:
    # Обрати правильну пряму
    x1, x2, y1 = sp.symbols('x_1 x_2 y_1')
    temp_F = [F[0],F[1]] # Коефіцієнти цільової функції F1
    temp_A_bound = [A_bound[0],A_bound[1]] # Коефіцієнти зв'язуючого обмеження
    eq_Y1 = sp.Eq(temp_A_bound[0]*x1 + temp_A_bound[1]*x2, y1)
    candidates=[list(A1[i])+[b1[i]] for i in range(len(res_1.slack)) if res_1.slack[i]<=0.0001]
    sols_C=[]
    best_der=-np.inf
    best_can=0
    for can in candidates:
        eq_c=sp.Eq(can[0]*x1+can[1]*x2,can[2])
        sol=sp.solve((eq_c,eq_Y1),(x1,x2))
        f1_C=temp_F[0]*sol[x1]+temp_F[1]*sol[x2]
        x1vs=sol[x1].evalf(subs={y1: c})
        x2vs=sol[x2].evalf(subs={y1: c})
        # print(x1vs)
        # print(x2vs)
        if x1vs>=0-eps and x2vs>=0-eps:
            if sp.nsimplify(sp.diff(temp_F[0]*sol[x1]+temp_F[1]*sol[x2],y1))>best_der:
                best_can=can
    #print(best_can)
    best_can=np.array(best_can)

    display(Math(fr"\text{{При збільшенні }}y_1>{c} \text{{ точка максимуму знаходться на перетині прямих }}{best_can[0]}x_1{text_sign(best_can[1])}{best_can[1]}x_2={best_can[2]}\ \ та\ \ {Ab_1[0]}x_1{text_sign(Ab_1[1])}{Ab_1[1]}x_2=y_1"))
    x1,x2,y1=sp.symbols('x_1 x_2 y_1')
    eq1=sp.Eq(best_can[0]*x1+best_can[1]*x2,best_can[2])
    eq2=sp.Eq(Ab_1[0]*x1+Ab_1[1]*x2,y1)
    sol=sp.solve((eq1,eq2),(x1,x2))
    x12=sol
    display(Math(r"\text{Розв'язуємо систему:}"))
    display(Math(fr"\begin{{cases}} {sp.latex(eq1.lhs)}={sp.latex(eq1.rhs)}\\{sp.latex(eq2.lhs)}={sp.latex(eq2.rhs)} \end{{cases}}"))
    display(Math(r"Звідси\ \ одержуємо"))
    temp=0
    counter=0
    for i in sol.keys():
        display(Math(fr"{i}={sp.together(sol[i])}"))
        temp=temp+F[counter]*sol[i]
        counter=counter+1
    
    display(Math(fr"f_1(y_1)={F[0]}x_1+{F[1]}x_2={sp.together(temp)}"))
    display(Math(fr"\text{{Це буде справедливо до точки }}А({res_1.x[0]};{res_1.x[1]})"))
    display(Math(fr"\text{{Їй відповідає }}y_1={-res_1.fun}"))
    display(Math(fr"\text{{В цій точці }}f_1(y_1)={sp.nsimplify(temp.evalf(subs={y1:-res_1.fun}))}"))
    f1.append((fr"{sp.together(temp)};\ \ {c} \leq y_1 \leq {-res_1.fun}"))
    f1.append((fr"{sp.nsimplify(temp.evalf(subs={y1:-res_1.fun}))};\ \ y_1 \geq {-res_1.fun}"))
    ders1.append(Fraction(F1[ind_1]/Ab_1[ind_1]).limit_denominator())
    ders1.append(sp.diff(temp,y1))
    ders1.append(sp.diff(sp.nsimplify(temp.evalf(subs={y1:-res_1.fun})),y1))
    ceilings1.append(c)
    ceilings1.append(-res_1.fun)
    ceilings1.append(np.inf)
else:
    best_can=0
    for i in range(len(res_1.slack)):
        if res_1.slack[i]==0 and i<=1:
            best_can=np.array(list(A1[i])+[b1[i]])
            display(Math(fr"\text{{При збільшенні }}y_1>{c} \text{{ точка максимуму знаходиться на перетині прямих }}{best_can[0]}x_1{text_sign(best_can[1])}{best_can[1]}x_2={best_can[2]}\ \ та\ \ {Ab_1[0]}x_1{text_sign(Ab_1[1])}{Ab_1[1]}x_2=y_1"))
    x1,x2,y1=sp.symbols('x_1 x_2 y_1')
    eq1=sp.Eq(best_can[0]*x1+best_can[1]*x2,best_can[2])
    eq2=sp.Eq(Ab_1[0]*x1+Ab_1[1]*x2,y1)
    sol=sp.solve((eq1,eq2),(x1,x2))
    x12=sol
    display(Math(r"\text{Розв'язуємо систему:}"))
    display(Math(fr"\begin{{cases}} {sp.latex(eq1.lhs)}={sp.latex(eq1.rhs)}\\{sp.latex(eq2.lhs)}={sp.latex(eq2.rhs)} \end{{cases}}"))
    display(Math(r"Звідси\ \ одержуємо"))
    temp=0
    counter=0
    for i in sol.keys():
        display(Math(fr"{i}={sp.together(sol[i])}"))
        temp=temp+F[counter]*sol[i]
        counter=counter+1
    
    display(Math(fr"f_1(y_1)={F[0]}x_1+{F[1]}x_2={sp.together(temp)}"))
    display(Math(fr"\text{{Це буде справедливо до точки }}А({res_1.x[0]};{res_1.x[1]})"))
    display(Math(fr"\text{{Їй відповідає }}y_1={-res_1.fun}"))
    display(Math(fr"\text{{В цій точці }}f_1(y_1)={sp.nsimplify(temp.evalf(subs={y1:-res_1.fun}))}"))
    f1.append((fr"{sp.together(temp)};\ \ {c} \leq y_1 \leq {-res_1.fun}"))
    f1.append((fr"{sp.nsimplify(temp.evalf(subs={y1:-res_1.fun}))};\ \ y_1 \geq {-res_1.fun}"))
    ders1.append(Fraction(F1[ind_1]/Ab_1[ind_1]).limit_denominator())
    ders1.append(sp.diff(temp,y1))
    ders1.append(sp.diff(sp.nsimplify(temp.evalf(subs={y1:-res_1.fun})),y1))
    ceilings1.append(c)
    ceilings1.append(-res_1.fun)
    ceilings1.append(np.inf)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [102]:
display(Math(r"\text{Таким чином отримуємо:}"))
display(Math(fr"f_1(y_1)=\begin{{cases}} {f1[0]}\\ {f1[1]}\\ {f1[2]}\\ \end{{cases}}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [103]:
temp_text=fr"""
    \begin{{cases}}
    max({F[2]}x_3{text_sign(F[3])}{F[3]}x_4)=f_2(y_2)\\
    {A_bound[2]}x_3{text_sign(A_bound[3])}{A_bound[3]}x_4 \leq y_2\\ 
    {A2[0][0]}x_3{text_sign(A2[0][1])}{A2[0][1]}x_4 \leq {b2[0]}\\ 
    {A2[1][0]}x_3{text_sign(A2[1][1])}{A2[1][1]}x_4 \leq {b2[1]}\\
    x_3 \geq 0,  x_4 \geq 0, y_2 \geq 0
    \end{{cases}}
"""
display(Math(r"\text{Запишемо 2-у блочну задачу}"))
display(Math(temp_text))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

# Block 2

In [104]:
F2=F[2:]
Ab_2=A_bound[2:]
coeffs_2=np.array([F2[i]/Ab_2[i] for i in range(len(F2))])
ind_2=np.argmax(coeffs_2)# яку з двох осей вибрали 0->OX, 1->OY
hsp2=np.hstack((A2,-1*b2.reshape(-1,1)))
interior_point=np.array([0.01,0.01])
hsp2_points=HalfspaceIntersection(halfspaces=hsp2, interior_point=interior_point)
# print(hsp1_points.intersections)
lim_OX2=0
lim_OY2=0
hp2=list(np.zeros_like(hsp2_points.intersections))
for p in range(len(hsp2_points.intersections)):
    hp2[p][0]=Fraction(hsp2_points.intersections[p][0]).limit_denominator()
    hp2[p][1]=Fraction(hsp2_points.intersections[p][1]).limit_denominator()
hp2=np.array(hp2)
for p in hp2:
    if p[0]==0 and p[1]!=0:
        lim_OY2=p
    if p[0]!=0 and p[1]==0:
        lim_OX2=p
H2=hsp2
V2=hp2
hull2 = ConvexHull(V2)
eps=1e-9
ordered_vertices2 = V2[hull2.vertices]
edges2 = []
m2 = H2.shape[0]

for i in range(len(ordered_vertices2)):
    p1 = ordered_vertices2[i]
    p2 = ordered_vertices2[(i+1) % len(ordered_vertices2)]

    for j in range(m2):
        a, b, c = H2[j]
        if abs(a*p1[0] + b*p1[1] + c) < eps and \
           abs(a*p2[0] + b*p2[1] + c) < eps:
            edges2.append({
                "edge": (p1, p2),
                "constraint_index": j,
                "line": H[j]
            })
            break
#display(edges)
display(Math(r"\text{Розв'язуємо її графоаналітично}"))
if ind_2==0:
    display(Math(r"\text{Тобто масимум досягається спочатку на вісі абсцис } x_3"))
    display(Math(fr"x_{{4}} = 0,\ \ x^0_3={lim_OX2[0]},\ \ y_2={Ab_2[ind_2]}x^0_3={Ab_2@lim_OX2}"))
    c=st(Ab_2@lim_OX2)
else:
    display(Math(r"\text{Тобто масимум досягається спочатку на вісі ординат } x_4"))
    display(Math(fr"x_{{3}} = 0,\ \ x^0_4={lim_OY2[1]},\ \ y_2={Ab_2[ind_2]}x^0_4={Ab_2@lim_OY2}"))
    c=st(Ab_2@lim_OY2)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [105]:
res_2=linprog(-Ab_2, A_ub=A2, b_ub=b2)
f2=[]
ders2=[]
ceilings2=[0]
x34=[]
f2.append((fr"{Fraction(F2[ind_2]/Ab_2[ind_2]).limit_denominator()}*y_2;\ \ 0 \leq y_2 \leq {c}"))
if len(hsp2_points.intersections)>3:
    # Обрати правильну пряму
    x3, x4, y2 = sp.symbols('x_3 x_4 y_2')
    temp_F = [F[2],F[3]] # Коефіцієнти цільової функції F1
    temp_A_bound = [A_bound[2],A_bound[3]] # Коефіцієнти зв'язуючого обмеження
    eq_Y2 = sp.Eq(temp_A_bound[0]*x3 + temp_A_bound[1]*x4, y2)
    candidates=[list(A2[i])+[b2[i]] for i in range(len(res_2.slack)) if res_2.slack[i]==0]
    sols_C=[]
    best_der=-np.inf
    best_can=0
    for can in candidates:
        eq_c=sp.Eq(can[0]*x3+can[1]*x4,can[2])
        sol=sp.solve((eq_c,eq_Y2),(x3,x4))
        f2_C=temp_F[0]*sol[x3]+temp_F[1]*sol[x4]
        x3vs=sol[x3].evalf(subs={y2: c})
        x4vs=sol[x4].evalf(subs={y2: c})
        if x3vs>=0-eps and x4vs>=0-eps:
            if sp.nsimplify(sp.diff(temp_F[0]*sol[x3]+temp_F[1]*sol[x4],y2))>best_der:
                best_can=can
    #print(best_can)
    best_can=np.array(best_can)

    display(Math(fr"\text{{При збільшенні }}y_2>{c} \text{{ точка максимуму знаходиться на перетині прямих }}{best_can[0]}x_3{text_sign(best_can[1])}{best_can[1]}x_4={best_can[2]}\ \ та\ \ {Ab_2[0]}x_3{text_sign(Ab_2[1])}{Ab_2[1]}x_4=y_2"))
    x3,x4,y2=sp.symbols('x_3 x_4 y_2')
    eq1=sp.Eq(best_can[0]*x3+best_can[1]*x4,best_can[2])
    eq2=sp.Eq(Ab_2[0]*x3+Ab_2[1]*x4,y2)
    sol=sp.solve((eq1,eq2),(x3,x4))
    x34=sol
    display(Math(r"\text{Розв'язуємо систему:}"))
    display(Math(fr"\begin{{cases}} {sp.latex(eq1.lhs)}={sp.latex(eq1.rhs)}\\{sp.latex(eq2.lhs)}={sp.latex(eq2.rhs)} \end{{cases}}"))
    display(Math(r"Звідси\ \ одержуємо"))
    temp=0
    counter=0
    for i in sol.keys():
        display(Math(fr"{i}={sp.together(sol[i])}"))
        temp=temp+F2[counter]*sol[i]
        counter=counter+1
    
    display(Math(fr"f_2(y_2)={F2[0]}x_3+{F2[1]}x_4={sp.together(temp)}"))
    display(Math(fr"\text{{Це буде справедливо до точки }}А({res_2.x[0]};{res_2.x[1]})"))
    display(Math(fr"\text{{Їй відповідає }}y_2={-res_2.fun}"))
    display(Math(fr"\text{{В цій точці }}f_2(y_2)={sp.nsimplify(temp.evalf(subs={y2:-res_2.fun}))}"))
    f2.append((fr"{sp.together(temp)};\ \ {c} \leq y_2 \leq {-res_2.fun}"))
    f2.append((fr"{sp.nsimplify(temp.evalf(subs={y2:-res_2.fun}))};\ \ y_2 \geq {-res_2.fun}"))
    ders2.append(Fraction(F2[ind_2]/Ab_2[ind_2]).limit_denominator())
    ders2.append(sp.diff(temp,y2))
    ders2.append(sp.diff(sp.nsimplify(temp.evalf(subs={y2:-res_2.fun})),y2))
    ceilings2.append(c)
    ceilings2.append(-res_2.fun)
    ceilings2.append(np.inf)
else:
    best_can=0
    for i in range(len(res_2.slack)):
        if res_2.slack[i]==0 and i<=1:
            best_can=np.array(list(A2[i])+[b2[i]])
            display(Math(fr"\text{{При збільшенні }}y_2>{c} \text{{ точка максимуму знаходться на перетині прямих }}{best_can[0]}x_3{text_sign(best_can[1])}{best_can[1]}x_4={best_can[2]}\ \ та\ \ {Ab_2[0]}x_3{text_sign(Ab_2[1])}{Ab_2[1]}x_4=y_2"))
    display(Math(fr"\text{{При збільшенні }}y_2>{c} \text{{ точка максимуму знаходться на перетині прямих }}{best_can[0]}x_3{text_sign(best_can[1])}{best_can[1]}x_4={best_can[2]}\ \ та\ \ {Ab_2[0]}x_3{text_sign(Ab_2[1])}{Ab_2[1]}x_4=y_2"))
    x3,x4,y2=sp.symbols('x_3 x_4 y_2')
    eq1=sp.Eq(best_can[0]*x3+best_can[1]*x4,best_can[2])
    eq2=sp.Eq(Ab_2[0]*x3+Ab_2[1]*x4,y2)
    sol=sp.solve((eq1,eq2),(x3,x4))
    x34=sol
    display(Math(r"\text{Розв'язуємо систему:}"))
    display(Math(fr"\begin{{cases}} {sp.latex(eq1.lhs)}={sp.latex(eq1.rhs)}\\{sp.latex(eq2.lhs)}={sp.latex(eq2.rhs)} \end{{cases}}"))
    display(Math(r"Звідси\ \ одержуємо"))
    temp=0
    counter=0
    for i in sol.keys():
        display(Math(fr"{i}={sp.together(sol[i])}"))
        temp=temp+F2[counter]*sol[i]
        counter=counter+1
    
    display(Math(fr"f_2(y_2)={F2[0]}x_3+{F2[1]}x_4={sp.together(temp)}"))
    display(Math(fr"\text{{Це буде справедливо до точки }}А({res_2.x[0]};{res_2.x[1]})"))
    display(Math(fr"\text{{Їй відповідає }}y_2={-res_2.fun}"))
    display(Math(fr"\text{{В цій точці }}f_2(y_2)={sp.nsimplify(temp.evalf(subs={y2:-res_2.fun}))}"))
    f2.append((fr"{sp.together(temp)};\ \ {c} \leq y_2 \leq {-res_2.fun}"))
    f2.append((fr"{sp.nsimplify(temp.evalf(subs={y2:-res_2.fun}))};\ \ y_2 \geq {-res_2.fun}"))
    ders2.append(Fraction(F2[ind_2]/Ab_2[ind_2]).limit_denominator())
    ders2.append(sp.diff(temp,y2))
    ders2.append(sp.diff(sp.nsimplify(temp.evalf(subs={y2:-res_2.fun})),y2))
    ceilings2.append(c)
    ceilings2.append(-res_2.fun)
    ceilings2.append(np.inf)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [106]:
display(Math(r"\text{Таким чином отримуємо:}"))
display(Math(fr"f_2(y_2)=\begin{{cases}} {f2[0]}\\ {f2[1]}\\ {f2[2]}\\ \end{{cases}}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [122]:
epoch_1=1
epoch_2=1
y1_vals=0
y2_vals=0
step=1
resurs=b_bound
# print(ceilings1)
# print(ceilings2)

def infty(x):
    if x==np.inf:
        return r'{\infty}'
    else:
        return x

while True:
    # print(epoch_1)
    # print(epoch_2)
    # print(resurs)  
    # print(y1_vals)  
    # print(y2_vals)
    # print("\n","\n","\n","\n","\n")  
    display(Math(fr"Крок\ \ {step}"))
    display(Math(fr"\frac{{df_1}}{{dy_1}}={ders1[epoch_1-1]},\ \ {ceilings1[epoch_1-1]} \leq y_1 \leq {infty(ceilings1[epoch_1])}"))
    display(Math(fr"\frac{{df_2}}{{dy_2}}={ders2[epoch_2-1]},\ \ {ceilings2[epoch_2-1]} \leq y_2 \leq {infty(ceilings2[epoch_2])}"))
    
    if ders1[epoch_1-1]>ders2[epoch_2-1]:
        display(Math(fr"\frac{{df_1}}{{dy_1}}>\frac{{df_2}}{{dy_2}}"))
        display(Math(fr"\text{{Віддаємо ресурс першій підсистемі}}"))
        add_res_1=min(resurs,ceilings1[epoch_1]-y1_vals)
        y1_vals=y1_vals+add_res_1
        display(Math(fr"y_1={y1_vals}"))
        # resurs=max(resurs-ceilings1[epoch_1],0)
        resurs=resurs-add_res_1
        epoch_1+=1
    else:
        display(Math(fr"\frac{{df_2}}{{dy_2}}>\frac{{df_1}}{{dy_1}}"))
        display(Math(fr"\text{{Віддаємо ресурс другій підсистемі}}"))
        add_res_2=min(resurs,ceilings2[epoch_2]-y2_vals)
        y2_vals=y2_vals+add_res_2
        display(Math(fr"y_2={y2_vals}"))
        # resurs=max(resurs-ceilings2[epoch_2],0)
        resurs=resurs-add_res_2
        epoch_2+=1
    step+=1
    # if epoch_2==3 and resurs!=0:
    #     display(Math(r"\text{Залишок ресурсу віддаємо першій підсистемі}"))
    #     y1_vals+=resurs
    #     resurs=0
    #     display(Math(fr"y_1={y1_vals}"))
    # if epoch_1==3 and resurs!=0:
    #     display(Math(r"\text{Залишок ресурсу віддаємо другій підсистемі}"))
    #     y2_vals+=resurs
    #     resurs=0
    #     display(Math(fr"y_2={y2_vals}"))
    if resurs==0 or epoch_1-1==len(ders1) or epoch_2-1==len(ders2):
        break
display(Math(r"\text{Остаточні значення:}"))
display(Math(fr"y_1={y1_vals}"))
display(Math(fr"y_2={y2_vals}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [109]:
tx1=sp.nsimplify(x12[x1].evalf(subs={y1: y1_vals}))
tx2=sp.nsimplify(x12[x2].evalf(subs={y1: y1_vals}))
tx3=sp.nsimplify(x34[x3].evalf(subs={y2: y2_vals}))
tx4=sp.nsimplify(x34[x4].evalf(subs={y2: y2_vals}))
display(Math(fr"Обчислюємо:"))
display(Math(fr"x_1={sp.together(x12[x1])}={tx1}"))
display(Math(fr"x_2={sp.together(x12[x2])}={tx2}"))
display(Math(fr"x_3={sp.together(x34[x3])}={tx3}"))
display(Math(fr"x_4={sp.together(x34[x4])}={tx4}"))
tx=np.array([tx1,tx2,tx3,tx4])

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [110]:
display(Math(r"\text{Перевіряємо правильність обчислень за зв'язуючим обмеженням}"))
display(Math(fr"{A_bound[0]}x_1{text_sign(A_bound[1])}{A_bound[1]}x_2{text_sign(A_bound[2])}{A_bound[2]}x_3{text_sign(A_bound[3])}{A_bound[3]}x_4={A_bound[0]}*{tx1}{text_sign(A_bound[1])}{A_bound[1]}*{tx2}{text_sign(A_bound[2])}{A_bound[2]}*{tx3}{text_sign(A_bound[3])}{A_bound[3]}*{tx4}={A_bound@tx}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [111]:
display(Math(r"\text{Тобто дане обмеження виконується строго. Таким чином, знайдене рішення є оптимальним}"))
display(Math(fr"\text{{Для нього max F}}={F[0]}*{tx1}{text_sign(F[1])}{F[1]}*{tx2}{text_sign(F[2])}{F[2]}*{tx3}{text_sign(F[3])}{F[3]}*{tx4}={F@tx}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Перевірка розподілу ресурсу у випадку рівності похідних

In [112]:
edge_1=(A1[0][0]*tx1+A1[0][1]*tx2<=b1[0])
edge_2=(A1[1][0]*tx1+A1[1][1]*tx2<=b1[1])
edge_3=(A2[0][0]*tx3+A2[0][1]*tx4<=b2[0])
edge_4=(A2[1][0]*tx3+A2[1][1]*tx4<=b2[1])
print(edge_1)
print(edge_2)
print(edge_3)
print(edge_4)

True
True
True
True


In [113]:
print(249/7)

35.57142857142857
